# 把资源地址发送给 Child Agent，就等于给它权限了吗？

## V0.8 Multi-Agent Runtime

这个 lab 渐进式构造 Parent/Child Agent 场景。你会看到 resource reference、IPC、ResourceShare、Capability delegation 分别解决不同问题。

**Core:** Reference != Permission. IPC != Authority. ResourceShare != Capability. Agent Tree != Process Tree.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "agentkernel").is_dir():
            return path
    raise RuntimeError("Run this notebook from inside the AgentKernel repository.")

REPOSITORY_ROOT = find_repo_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))
LABS_ROOT = REPOSITORY_ROOT / "examples" / "labs"
if str(LABS_ROOT) not in sys.path:
    sys.path.insert(0, str(LABS_ROOT))

from lab_helpers import event_rows, grant_rows, print_table, process_row, trajectory

## 1. Parent has authority; Child starts without authority

In [ ]:
import asyncio
import tempfile
from collections.abc import Mapping
from pathlib import Path

from agentkernel import (
    AgentRegistry, CapabilityEvaluator, CapabilityGrant, DelegateCapabilityRequest,
    ErrorCode, InMemoryIPCPersistence, KernelIPC, LocalResourceStore, ProcessManager,
    RESOURCE_READ_ACTION, ResourceAccessDenied, ResourceOwner, ResourceService,
    ResourceShareRegistry, Session, TOOL_EXECUTE_ACTION, ToolCall, ToolDefinition,
    ToolExecutionContext, ToolRegistry, ToolSchema,
)
from agentkernel.protocol import JsonValue

def evaluator_for(agent):
    return CapabilityEvaluator.from_agent_capabilities(
        agent_id=agent.agent_id,
        capabilities=agent.capabilities,
        capability_grants=agent.capability_grants,
    )

async def add(arguments: Mapping[str, JsonValue], _context: ToolExecutionContext) -> JsonValue:
    return int(arguments["left"]) + int(arguments["right"])

agents = AgentRegistry()
parent_session = Session("lab-v0-8-parent-session")
child_session = Session("lab-v0-8-child-session")
parent = agents.create_root(
    agent_id="agent-parent",
    session=parent_session,
    capability_grants=(
        CapabilityGrant("agent-parent", TOOL_EXECUTE_ACTION, "tool://math.add"),
        CapabilityGrant("agent-parent", RESOURCE_READ_ACTION, "artifact://**"),
    ),
    creation_id="create-parent",
)
child = agents.create_child(
    parent_agent_id=parent.control.agent_id,
    agent_id="agent-child",
    session=child_session,
    creation_id="create-child",
    record_session=parent_session,
)
processes = ProcessManager(agent_registry=agents)
parent_process = processes.create_process(process_id="process-parent", agent=parent.control)
child_process = processes.create_child_process(parent_process_id="process-parent", process_id="process-child", agent=child.control)
print_table([
    {"tree": "Agent lineage", "value": "/".join(agents.lineage("agent-child"))},
    {"tree": "Process lineage", "value": "/".join(processes.lineage("process-child"))},
    {"tree": "same tree?", "value": agents.lineage("agent-child") == processes.lineage("process-child")},
    {"tree": "child grants", "value": len(child.control.capability_grants)},
])

## 2. Child proposes a Tool call before delegation: DENY

In [ ]:
tools = ToolRegistry()
tools.register(ToolDefinition(
    schema=ToolSchema("math.add", "Add two integers.", {"type": "object"}),
    handler=add,
    required_action=TOOL_EXECUTE_ACTION,
    required_resource="tool://math.add",
))
call = ToolCall("call-add-1", "math.add", {"left": 2, "right": 3})
before_tool = asyncio.run(tools.execute(call, child.control))
print_table([
    {"attempt": "child tool before delegation", "ok": before_tool.ok, "result": before_tool.error.code.value if before_tool.error else before_tool.output},
])
assert before_tool.error is not None and before_tool.error.code is ErrorCode.EACCES

## 3. Send Resource URI via IPC: reference arrives, authority does not

In [ ]:
tmpdir = tempfile.TemporaryDirectory(prefix="agentkernel-lab-v0-8-")
shares = ResourceShareRegistry(agent_registry=agents, clock=lambda: 100.0)
resources = ResourceService(
    LocalResourceStore(Path(tmpdir.name) / "resources"),
    share_registry=shares,
    resource_id_factory=lambda: "res_secret",
    handle_id_factory=lambda: "hdl_secret",
    clock=lambda: 10.0,
)
owner = ResourceOwner(parent.control.agent_id, parent.control.session_id)
child_owner = ResourceOwner(child.control.agent_id, child.control.session_id)
handle = resources.create_artifact(
    b"secret-bytes",
    owner=owner,
    media_type="text/plain",
    encoding="utf-8",
    source_tool_name="producer",
    source_tool_call_id="call-producer",
    source_operation_id="op-producer",
)
ipc = KernelIPC(
    agent_registry=agents,
    process_manager=processes,
    sessions={parent.control.agent_id: parent_session, child.control.agent_id: child_session},
    persistence=InMemoryIPCPersistence(),
    time_fn=lambda: 1.0,
)
ipc.create_channel(
    channel_id="channel-parent-child",
    sender_agent_id=parent.control.agent_id,
    receiver_agent_id=child.control.agent_id,
    receiver_process_id=child_process.process_id,
)
ipc.send(
    channel_id="channel-parent-child",
    sender_process_id=parent_process.process_id,
    payload={"body": "use this artifact"},
    resource_refs=(handle.uri,),
    message_id="message-resource-ref",
    correlation_id="corr-message",
)
delivered = ipc.receive(
    channel_id="channel-parent-child",
    receiver_agent_id=child.control.agent_id,
    receiver_process_id=child_process.process_id,
)
ref_allows = True
try:
    resources.read(
        handle.uri,
        owner=child_owner,
        capability_evaluator=CapabilityEvaluator((CapabilityGrant(child.control.agent_id, RESOURCE_READ_ACTION, handle.uri),)),
    )
except ResourceAccessDenied:
    ref_allows = False
print_table([
    {"fact": "delivered reference", "value": delivered.resource_refs[0]},
    {"fact": "reference alone grants access", "value": ref_allows},
])

## 4. ResourceShare without current capability: still DENY

In [ ]:
share = resources.share(
    handle.uri,
    owner=owner,
    grantee_agent_id=child.control.agent_id,
    allowed_actions=(RESOURCE_READ_ACTION,),
    record_session=parent_session,
    share_id="share_secret",
    correlation_id="corr-share",
)
share_without_capability = True
try:
    resources.read(handle.uri, owner=child_owner, capability_evaluator=evaluator_for(agents.get(child.control.agent_id)))
except ResourceAccessDenied:
    share_without_capability = False
print_table([
    {"fact": "share created", "value": share.allowed},
    {"fact": "share without capability grants access", "value": share_without_capability},
])

## 5. Delegate narrowed resource and tool capabilities: ALLOW

In [ ]:
resource_decision = agents.delegate_capability(
    DelegateCapabilityRequest("agent-parent", "agent-child", RESOURCE_READ_ACTION, handle.uri, correlation_id="delegate-resource"),
    record_session=child_session,
)
resource_read = resources.read(handle.uri, owner=child_owner, capability_evaluator=evaluator_for(agents.get("agent-child")))
tool_decision = agents.delegate_capability(
    DelegateCapabilityRequest("agent-parent", "agent-child", TOOL_EXECUTE_ACTION, "tool://math.add", correlation_id="delegate-math"),
    record_session=child_session,
)
after_tool = asyncio.run(tools.execute(call, agents.get("agent-child")))
print_table([
    {"attempt": "resource after share + capability", "allowed": resource_decision.allowed, "result": resource_read.data.decode()},
    {"attempt": "tool after capability", "allowed": tool_decision.allowed, "result": after_tool.output},
])
tmpdir.cleanup()

## 6. Access matrix

In [ ]:
print_table([
    {"reference": "no", "share": "no", "capability": "no", "result": "DENY"},
    {"reference": "yes", "share": "no", "capability": "yes", "result": "DENY"},
    {"reference": "yes", "share": "yes", "capability": "no", "result": "DENY"},
    {"reference": "yes", "share": "yes", "capability": "yes", "result": "ALLOW"},
])
trajectory("Parent owns resource", "IPC sends reference", "ResourceShare records sharing", "Capability delegation narrows authority", "Child access allowed")

## Invariant

Child agents do not inherit parent authority by accident. Cross-agent resource access needs both sharing state and current capability authorization.

## WHAT THIS DEMONSTRATES / 本实验验证什么

- Agent Tree and Process Tree are separate runtime structures.
- IPC resource references do not grant authority by themselves.
- ResourceShare and narrowed delegation are both required for resource access.
- Tool authority also requires explicit capability delegation.

## WHAT THIS DOES NOT DEMONSTRATE / 本实验不证明什么

- It does not implement V0.9 memory.
- It does not prove distributed multi-agent correctness.
- It does not prove production sandbox security.